# TrustGate session 1 on Kaggle (125M, free)

Two modes, set in the first code cell:

- **`MODE = "smoke"`** needs **no data and no checkpoint.** It runs experiment 003 (random-init
  125M through the real vendor model, including `carry-is-non-trivial`) and the random-init
  `--gate-eval` smoke. It answers whether the session fits on a Kaggle GPU. **Run this first.**
- **`MODE = "session"`** is the whole of runbook Parts C1 to C5 on the real 125M checkpoint:
  000 baseline, C1b prep, C2 timing, C3 kill gate, C4 sequence arms and C5 gate measurement.

**Notebook settings (right-hand panel):**
1. **Accelerator: GPU T4 x2.** The notebook pins GPU 0 (`CUDA_VISIBLE_DEVICES=0`). Without the
   pin, the vendor's `ModelSharding` builds its mesh from `jax.device_count()` and spreads
   the model over both T4s as 2-way data parallel, which is not the single-device setup the
   pre-registration assumes. Use T4 rather than P100: sm_75 is safely covered by the
   CUDA 12.8 / cuDNN 9 wheels, and Pascal is not guaranteed to be. A T4 has 16 GB and no
   native bf16, so this notebook sets `COMPUTE_DTYPE=fp32`. That dtype is recorded in every
   report, as the 2026-09-22 revision requires.
2. **Internet: On.** This needs a phone-verified Kaggle account. The clone, `uv sync` and the
   GPT-2 fluency reference all download.
3. **Session mode only:** add a private Kaggle **Dataset** holding the two tars from
   `scripts/colab/fetch_checkpoint_colab.ipynb` (`ttt-handoff-<CKPT>.tar`,
   `ttt-books3-val.tar`). Kaggle may auto-extract them, and either form works. Add the
   **Secrets** `WANDB_ENTITY`, `WANDB_PROJECT` and `WANDB_KEY` (Add-ons → Secrets).

**Running unattended:** use **Save Version → Save & Run All (Commit)**. It keeps running with
the browser closed, for up to 12 hours. Results are copied to `/kaggle/working/results` after
every step, so a timeout keeps whatever finished. To resume after a timeout, attach this
notebook's previous output as an input and set `RESUME_FROM`. C3 to C5 resume from their
ledgers (runbook, "Every artifact is written to disk as it is produced").

Nothing here renders a verdict except C3, and C3 renders one only against the pre-registered
bars, unchanged.

In [ ]:
# ---- settings ---------------------------------------------------------------
MODE = "smoke"              # "smoke" (no data, run first) or "session"
BRANCH = "infra/gpu-session-1"
CKPT = "125m_ttt_e2e_finetune_books_8k_1x_cc"
COMPUTE_DTYPE = "fp32"      # T4/P100 have no native bf16 (runbook Part B-kaggle)
DATASET_DIR = ""            # session mode: e.g. "/kaggle/input/ttt-session1-data"
MAX_ITERS = None            # session mode: set from the C2 timing before C3 runs (int)
RESUME_FROM = ""            # e.g. "/kaggle/input/<previous-output>/results" to resume C3-C5
REPO_URL = "https://github.com/Manas-Maahir/Trust-Gated-Fast-Weight-Updates-for-TTT-E2E-LLMs.git"

import os, subprocess, shutil, pathlib
os.environ.update({
    "MODE": MODE, "BRANCH": BRANCH, "CKPT": CKPT, "COMPUTE_DTYPE": COMPUTE_DTYPE, "REPO_URL": REPO_URL,
    "REPO": "/tmp/TTT",                       # off /kaggle/working: the vendor env is ~10 GB
    "DATA_ROOT": "/tmp/ttt-data",
    "EXP_DIR": "/tmp/ttt-runs",
    "OUT": "/kaggle/working/results",
    "UV_CACHE_DIR": "/tmp/uv-cache",
    "XLA_PYTHON_CLIENT_MEM_FRACTION": "0.92",
    "CUDA_VISIBLE_DEVICES": "0",              # one GPU: the vendor mesh spans every visible device
    "PATH": f"/root/.local/bin:{os.environ['PATH']}",
})
pathlib.Path("/kaggle/working/results").mkdir(parents=True, exist_ok=True)
print("MODE =", MODE)

In [ ]:
%%bash
# ---- GPU, clone, uv -----------------------------------------------------------
set -euo pipefail
nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
if [ ! -d "$REPO/.git" ]; then
  git clone -q --recursive -b "$BRANCH" "$REPO_URL" "$REPO"
fi
git -C "$REPO" log --oneline -1
command -v uv >/dev/null || pip install -q uv
uv --version
df -h /tmp /kaggle/working | tail -2

## Smoke mode: will the session fit on this GPU?

The same two checks the laptop plan needs (`COST_MODEL.md` §9.4.2), on 16 GB instead of 8 GB.
They need no checkpoint and no data, and they render no verdict.

In [ ]:
%%bash
# ---- vendor env (the bootstrap's own Step 2) plus the fluency deps --------------
set -euo pipefail
[ "${MODE:-}" = "session" ] && { echo "session mode: the bootstrap builds this env; skipping"; exit 0; } || true
cd "$REPO/vendor/ttt-e2e" && uv sync --frozen
VPY="$REPO/vendor/ttt-e2e/.venv/bin/python"
uv pip install --python "$VPY" -q "safetensors>=0.4" "tokenizers>=0.19"
"$VPY" -c "import jax; print(jax.devices())"

In [ ]:
import subprocess, os
if MODE == "smoke":
    VPY = f"{os.environ['REPO']}/vendor/ttt-e2e/.venv/bin/python"
    env = dict(os.environ, PYTHONPATH=f"{os.environ['REPO']}/src")
    for seq in (8192, 4096):
        out = f"/kaggle/working/results/smoke-{seq}.json"
        print(f"\n===== experiment 003 at seq_length={seq}, compute_dtype={COMPUTE_DTYPE} =====", flush=True)
        r = subprocess.run([VPY, "experiments/003-smoke-125m/run_smoke.py", "--seq-length", str(seq),
                            "--compute-dtype", COMPUTE_DTYPE, "--out", out], cwd=os.environ["REPO"], env=env)
        print(f"exit {r.returncode}; report: {out}")
        if r.returncode == 0:
            break   # 8192 passed: 4096 adds nothing

In [ ]:
%%bash
# ---- gate smoke: the same command prepare_phase1.sh step 5 runs, plus the dtype --
set -uo pipefail
[ "${MODE:-}" = "session" ] && { echo "session mode: prepare_phase1.sh runs this; skipping"; exit 0; }
cd "$REPO"
VPY="$REPO/vendor/ttt-e2e/.venv/bin/python"
"$VPY" scripts/fetch_reference_model.py
PYTHONPATH=src "$VPY" -m trustgate.eval.cli \
  --objective degrade --strategy select --gate-eval \
  --random-init --size 125m --compute-dtype "$COMPUTE_DTYPE" --stream-tokens 8192 --seeds 0 1 \
  --overhead-repeats 3 --out "$OUT/gate-smoke" 2>&1 | tee "$OUT/gate-smoke.log" | tail -40
echo "gate smoke exit: ${PIPESTATUS[0]}"

In [ ]:
import json, glob
for f in sorted(glob.glob("/kaggle/working/results/smoke-*.json")):
    d = json.load(open(f))
    cfg = d.get("config", {})
    print(f"\n{f}  (seq_length={cfg.get('seq_length')}, compute_dtype={cfg.get('compute_dtype')})")
    for c in d["checks"]:
        print(f"  {'PASS' if c['passed'] else 'FAIL'}  {c['name']:<24} {str(c['detail'])[:110]}")
print("\nRead: carry-is-non-trivial must PASS and random-init-loss must not be nan.")
print("Peak memory is not sampled here; if both pass at 8192 the session fits this card.")

## Session mode: runbook C1 to C5

The cells below do nothing unless `MODE = "session"`. Each one is a runbook step. Read that
step's text in `docs/protocols/gpu-session-1-runbook.md` before trusting its output.

In [ ]:
# ---- data from the attached Dataset into writable /tmp (reshape rewrites zarr metadata) --
import os, glob, shutil, subprocess, tarfile
if MODE == "session":
    assert DATASET_DIR and os.path.isdir(DATASET_DIR), "set DATASET_DIR to the attached Dataset"
    root = os.environ["DATA_ROOT"]; os.makedirs(root, exist_ok=True)
    for t in glob.glob(f"{DATASET_DIR}/**/*.tar", recursive=True):
        print("extracting", t); tarfile.open(t).extractall(root)
    for name in ("ttt-handoff", "llama3-books3", "train-zarr"):   # auto-extracted by Kaggle
        hits = [p for p in glob.glob(f"{DATASET_DIR}/**/{name}", recursive=True) if os.path.isdir(p)]
        if hits and not os.path.exists(f"{root}/{name}"):
            print("copying", hits[0]); shutil.copytree(hits[0], f"{root}/{name}")
    for need in (f"ttt-handoff/{CKPT}", f"ttt-handoff/checkpoint-sha256-{CKPT}.txt", "llama3-books3", "train-zarr/train"):
        assert os.path.exists(f"{root}/{need}"), f"missing {root}/{need}"
    print("data ready under", root)
    from kaggle_secrets import UserSecretsClient
    s = UserSecretsClient()
    for k in ("WANDB_ENTITY", "WANDB_PROJECT", "WANDB_KEY"):
        os.environ[k] = s.get_secret(k)
    print("W&B secrets loaded (values not printed)")

In [ ]:
%%bash
# ---- B3-kaggle: bootstrap ------------------------------------------------------
set -euo pipefail
[ "${MODE:-}" = "session" ] || { echo "smoke mode: skipping"; exit 0; }
cd "$REPO"
CKPT_DIR="$DATA_ROOT/ttt-handoff/$CKPT" \
CKPT_SHA_MANIFEST="$DATA_ROOT/ttt-handoff/checkpoint-sha256-$CKPT.txt" \
  bash scripts/bootstrap_gpu_box.sh 2>&1 | tail -60

In [ ]:
%%bash
# ---- C1: 000 baseline (S1-S4 only at 125M) --------------------------------------
set -uo pipefail
[ "${MODE:-}" = "session" ] || { echo "smoke mode: skipping"; exit 0; }
cd "$REPO"
DEADLINE_HOURS=3 bash scripts/run_gpu_session.sh 2>&1 | tail -60
echo "C1 exit: ${PIPESTATUS[0]}   (0 PASS, 1 FAIL, 2 incomplete -- do not proceed past a FAIL)"
mkdir -p "$OUT/000" && cp -r experiments/000-repro-baseline/results/session-* "$OUT/000/" 2>/dev/null; true

In [ ]:
%%bash
# ---- C1b: prepare for 001 and the gate (TRAIN_DIR: no billing project) ----------
set -euo pipefail
[ "${MODE:-}" = "session" ] || { echo "smoke mode: skipping"; exit 0; }
cd "$REPO"
TRAIN_DIR="$DATA_ROOT/train-zarr" bash scripts/prepare_phase1.sh 2>&1 | tail -40

In [ ]:
# ---- resume: restore C3-C5 ledgers from a previous version's output --------------
import os, shutil
if MODE == "session" and RESUME_FROM:
    R = f"{os.environ['REPO']}/experiments/001-attack-spike/results"
    src = f"{RESUME_FROM}/001"
    assert os.path.isdir(src), f"no {src}"
    shutil.copytree(src, R, dirs_exist_ok=True)
    print("restored", src, "->", R)

In [ ]:
%%bash
# ---- C2: time one adapt-and-eval, then set MAX_ITERS in the settings cell ---------
set -uo pipefail
[ "${MODE:-}" = "session" ] || { echo "smoke mode: skipping"; exit 0; }
source "$EXP_DIR/phase1.env"
time $TG --objective degrade --strategy select \
  --checkpoint $CKPT_DEST --checkpoint-manifest $CKPT_MANIFEST \
  --corpus-file $T/train.npy --eval-file $T/val.npy \
  --size $SIZE $DTYPE_FLAG --seq-length 8192 --stream-tokens 8192 \
  --seeds 0 1 --max-iters 1 --early-stop-patience 0 --no-resume \
  --out $EXP_DIR/c2-timing 2>&1 | tee -a $R/logs/c2-timing.log | grep -E "adapt-and-eval|RESULT|Error"
echo
echo "Choose MAX_ITERS so that 5 x (MAX_ITERS + 3) x per-eval seconds fits the time left (runbook C2)."
mkdir -p "$OUT/001" && cp -r $R/. "$OUT/001/"

In [ ]:
# ---- C3: 001 kill gate -- the verdict ---------------------------------------------
import os, subprocess
if MODE == "session":
    assert isinstance(MAX_ITERS, int) and MAX_ITERS > 0, "set MAX_ITERS from the C2 timing first"
    cmd = f'''source "$EXP_DIR/phase1.env"
$TG --objective degrade --strategy select \
  --checkpoint $CKPT_DEST --checkpoint-manifest $CKPT_MANIFEST \
  --corpus-file $T/train.npy --eval-file $T/val.npy \
  --corpus-split train --eval-split val \
  --size $SIZE $DTYPE_FLAG --seq-length 8192 --stream-tokens 8192 \
  --seeds 0 1 2 3 4 --max-iters {MAX_ITERS} \
  --out $R 2>&1 | tee -a $R/logs/c3-spike.log | tail -60
mkdir -p "$OUT/001" && cp -r $R/. "$OUT/001/"'''
    subprocess.run(["bash", "-c", cmd], check=False)

In [ ]:
%%bash
# ---- C4 (secondary, non-gating) and C5 (gate measurement) -------------------------
set -uo pipefail
[ "${MODE:-}" = "session" ] || { echo "smoke mode: skipping"; exit 0; }
source "$EXP_DIR/phase1.env"
if [ -f "$R/arms.pkl" ]; then
  $TG --objective degrade --strategy select --sequence-eval \
    --checkpoint $CKPT_DEST --checkpoint-manifest $CKPT_MANIFEST \
    --corpus-file $T/train.npy --eval-file $T/val.npy \
    --arms-file $R/arms.pkl \
    --size $SIZE $DTYPE_FLAG --seq-length 8192 --seeds 0 1 2 3 4 \
    --out $R/sequence 2>&1 | tee -a $R/logs/c4-sequence.log | tail -20
  ARMS="--arms-file $R/arms.pkl"
else
  echo "C3 wrote no arms.pkl (NULL RESULT): skipping C4; C5 uses --uncrafted-arms (runbook C5)."
  ARMS="--uncrafted-arms"
fi
mkdir -p "$OUT/001" && cp -r $R/. "$OUT/001/"
$TG --objective degrade --strategy select --gate-eval \
  --checkpoint $CKPT_DEST --checkpoint-manifest $CKPT_MANIFEST \
  --corpus-file $T/train.npy --eval-file $T/val.npy --probe-file $T/probe.npy \
  $ARMS \
  --size $SIZE $DTYPE_FLAG --seq-length 8192 --seeds 0 1 2 3 4 \
  --gate-quantiles 0.9 0.99 \
  --out $R/gate 2>&1 | tee -a $R/logs/c5-gate.log | tail -30
cp -r $R/. "$OUT/001/"
echo "Done. Everything is under /kaggle/working/results. Download it from the Output tab."